# NeuroRes-GNN: Modular ML Pipeline
## Brain Graph Super-Resolution End-to-End

Clean, reproducible pipeline with modular components that can be extracted to separate files.

## Part 0: Environment Setup

In [ ]:
import sys
from pathlib import Path

# Configure paths for imports
notebook_dir = Path('.').resolve()
repo_root = notebook_dir.parent

# Add necessary paths
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / 'gcn-encoder-ca-decoder'))

print(f"Notebook directory: {notebook_dir}")
print(f"Repository root: {repo_root}")
print(f"Python paths configured.")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from typing import Tuple, Dict, Any
import warnings

warnings.filterwarnings('ignore')

# Set device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

## Part 1: Data Module

Handles loading, validation, and preprocessing of datasets.

In [ ]:
class DataModule:
    """Encapsulates all data operations."""
    
    def __init__(self, repo_root: Path, data_dir: str = 'data'):
        self.repo_root = Path(repo_root)
        self.data_dir = self.repo_root / data_dir
        self.validate_data_dir()
    
    def validate_data_dir(self) -> None:
        """Check if data directory exists."""
        if not self.data_dir.exists():
            raise FileNotFoundError(f"Data directory not found: {self.data_dir}")
        print(f"✓ Data directory verified: {self.data_dir}")
    
    @staticmethod
    def load_csv(path: Path) -> np.ndarray:
        """Load CSV file, skip header row."""
        arr = np.loadtxt(path, delimiter=",", skiprows=1, dtype=np.float32)
        if arr.ndim == 1:
            arr = arr[None, :]
        return arr
    
    def load_datasets(self) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Load training and test datasets."""
        X_lr_train = self.load_csv(self.data_dir / 'lr_train.csv')
        Y_hr_train = self.load_csv(self.data_dir / 'hr_train.csv')
        X_lr_test = self.load_csv(self.data_dir / 'lr_test.csv')
        
        return X_lr_train, Y_hr_train, X_lr_test
    
    def get_shapes(self) -> Dict[str, Tuple]:
        """Return dataset shapes without loading full data."""
        X_train, Y_train, X_test = self.load_datasets()
        return {
            'X_lr_train': X_train.shape,
            'Y_hr_train': Y_train.shape,
            'X_lr_test': X_test.shape,
        }


# Initialize data module
data_module = DataModule(repo_root)
shapes = data_module.get_shapes()

print("\nDataset Shapes:")
for name, shape in shapes.items():
    print(f"  {name}: {shape}")

In [ ]:
# Load all data
X_lr_train, Y_hr_train, X_lr_test = data_module.load_datasets()
print(f"\n✓ All data loaded successfully")

## Part 2: Preprocessing Module

Matrix vectorization, tensor conversion, and feature extraction.

In [ ]:
# Import preprocessing utilities from gcn-encoder-ca-decoder
from data_utils import vec_to_adj, adj_to_vec, lr_node_features, to_tensor

# Import matrix vectorizer from utils
from utils.matrix_vectorizer import MatrixVectorizer

print("✓ Preprocessing utilities imported")

In [ ]:
class PreprocessingModule:
    """Encapsulates preprocessing operations."""
    
    def __init__(self, n_lr: int = 160, n_hr: int = 268, device: str = 'cpu'):
        self.n_lr = n_lr
        self.n_hr = n_hr
        self.device = device
    
    def vectorize(self, matrix: np.ndarray) -> np.ndarray:
        """Convert symmetric matrix to upper-triangle vector."""
        return MatrixVectorizer.vectorize(matrix, include_diagonal=False)
    
    def anti_vectorize(self, vector: np.ndarray, size: int) -> np.ndarray:
        """Convert upper-triangle vector to symmetric matrix."""
        return MatrixVectorizer.anti_vectorize(vector, size, include_diagonal=False)
    
    def vec_batch_to_adj(self, vec_batch: torch.Tensor, n: int) -> torch.Tensor:
        """Convert batch of vectors to adjacency matrices."""
        return vec_to_adj(vec_batch, n)
    
    def adj_batch_to_vec(self, adj_batch: torch.Tensor) -> torch.Tensor:
        """Convert batch of adjacency matrices to vectors."""
        return adj_to_vec(adj_batch)
    
    def get_node_features(self, adj_batch: torch.Tensor) -> torch.Tensor:
        """Extract node features from adjacency matrices."""
        return lr_node_features(adj_batch)
    
    def numpy_to_tensor(self, x: np.ndarray) -> torch.Tensor:
        """Convert numpy array to tensor."""
        return to_tensor(x, self.device)


# Initialize preprocessing module
preprocessor = PreprocessingModule(n_lr=160, n_hr=268, device=DEVICE)
print("✓ Preprocessing module initialized")

In [ ]:
# Demo: Vectorization check
sample_vec = X_lr_train[0]
sample_adj = preprocessor.anti_vectorize(sample_vec, 160)
sample_vec_recon = preprocessor.vectorize(sample_adj)

print(f"Vectorization Demo:")
print(f"  Original vector shape: {sample_vec.shape}")
print(f"  → Adjacency: {sample_adj.shape}")
print(f"  → Vector: {sample_vec_recon.shape}")
print(f"  Reconstruction error: {np.linalg.norm(sample_vec - sample_vec_recon):.2e}")
print(f"  Symmetric: {np.allclose(sample_adj, sample_adj.T)}")

## Part 3: Model Module

Model configuration and initialization.

In [ ]:
# Import model from gcn-encoder-ca-decoder
from model import LR2HRGenerator

print("✓ Model imported")

In [ ]:
class ModelConfig:
    """Model hyperparameters."""
    
    def __init__(self):
        self.n_lr = 160
        self.n_hr = 268
        self.d_model = 64
        self.gcn_layers = 2
        self.attn_heads = 4
        self.dropout = 0.1
        self.in_node_feat_dim = 2
    
    def to_dict(self) -> Dict[str, Any]:
        return self.__dict__
    
    def __repr__(self) -> str:
        return f"ModelConfig({self.to_dict()})"


class ModelModule:
    """Encapsulates model operations."""
    
    def __init__(self, config: ModelConfig, device: str):
        self.config = config
        self.device = device
        self.model = None
    
    def build(self) -> nn.Module:
        """Instantiate model."""
        self.model = LR2HRGenerator(
            n_lr=self.config.n_lr,
            n_hr=self.config.n_hr,
            d_model=self.config.d_model,
            gcn_layers=self.config.gcn_layers,
            attn_heads=self.config.attn_heads,
            dropout=self.config.dropout,
            in_node_feat_dim=self.config.in_node_feat_dim,
        ).to(self.device)
        return self.model
    
    def get_parameters_count(self) -> int:
        """Count model parameters."""
        if self.model is None:
            raise RuntimeError("Model not built. Call build() first.")
        return sum(p.numel() for p in self.model.parameters())
    
    def create_optimizer(self, lr: float, weight_decay: float) -> torch.optim.Optimizer:
        """Create AdamW optimizer."""
        if self.model is None:
            raise RuntimeError("Model not built. Call build() first.")
        return torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=weight_decay)


# Initialize model module
model_config = ModelConfig()
model_module = ModelModule(model_config, DEVICE)
model = model_module.build()
param_count = model_module.get_parameters_count()

print(f"\nModel Configuration:")
print(f"  {model_config}")
print(f"\nModel Summary:")
print(f"  Total parameters: {param_count:,}")
print(f"  Device: {DEVICE}")

## Part 4: Training Module

Cross-validation training loop with checkpointing.

In [ ]:
class TrainingConfig:
    """Training hyperparameters."""
    
    def __init__(self):
        self.epochs = 50
        self.batch_size = 8
        self.lr = 1e-3
        self.weight_decay = 1e-5
        self.n_folds = 3
        self.random_state = 42
    
    def to_dict(self) -> Dict[str, Any]:
        return self.__dict__


class TrainingModule:
    """Handles training pipeline."""
    
    def __init__(
        self,
        model: nn.Module,
        preprocessor: PreprocessingModule,
        config: TrainingConfig,
        device: str,
    ):
        self.model = model
        self.preprocessor = preprocessor
        self.config = config
        self.device = device
        self.loss_fn = nn.MSELoss()
        self.history = {'train_loss': [], 'val_loss': []}
    
    def train_epoch(
        self,
        loader: DataLoader,
        optimizer: torch.optim.Optimizer,
    ) -> float:
        """Train for one epoch."""
        self.model.train()
        losses = []
        
        for x_vec, y_vec in loader:
            A = self.preprocessor.vec_batch_to_adj(x_vec, self.preprocessor.n_lr)
            X_nodes = self.preprocessor.get_node_features(A)
            
            pred = self.model(A, X_nodes)
            loss = self.loss_fn(pred, y_vec)
            
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            
            losses.append(loss.item())
        
        return float(np.mean(losses))
    
    @torch.no_grad()
    def validate(
        self,
        loader: DataLoader,
    ) -> float:
        """Validate on loader."""
        self.model.eval()
        losses = []
        
        for x_vec, y_vec in loader:
            A = self.preprocessor.vec_batch_to_adj(x_vec, self.preprocessor.n_lr)
            X_nodes = self.preprocessor.get_node_features(A)
            
            pred = self.model(A, X_nodes)
            loss = self.loss_fn(pred, y_vec)
            
            losses.append(loss.item())
        
        return float(np.mean(losses))
    
    def train_fold(
        self,
        X_train: torch.Tensor,
        Y_train: torch.Tensor,
        X_val: torch.Tensor,
        Y_val: torch.Tensor,
        fold_id: int = 1,
    ) -> Dict[str, Any]:
        """Train for one fold."""
        print(f"\n{'='*60}")
        print(f"Fold {fold_id} Training")
        print(f"{'='*60}")
        
        # Create dataloaders
        train_loader = DataLoader(
            TensorDataset(X_train, Y_train),
            batch_size=self.config.batch_size,
            shuffle=True,
        )
        val_loader = DataLoader(
            TensorDataset(X_val, Y_val),
            batch_size=self.config.batch_size,
            shuffle=False,
        )
        
        # Setup
        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.config.lr,
            weight_decay=self.config.weight_decay,
        )
        
        best_val_loss = float('inf')
        best_state = None
        
        # Training loop
        for epoch in range(1, self.config.epochs + 1):
            train_loss = self.train_epoch(train_loader, optimizer)
            val_loss = self.validate(val_loader)
            
            improved = val_loss < best_val_loss
            if improved:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
            
            status = " ✓" if improved else ""
            if epoch % 10 == 0 or epoch == 1:
                print(
                    f"Epoch {epoch:3d}/{self.config.epochs} | "
                    f"Train: {train_loss:.6f} | Val: {val_loss:.6f}{status}"
                )
        
        # Restore best model
        self.model.load_state_dict(best_state)
        
        print(f"\nFold {fold_id} Best Val Loss: {best_val_loss:.6f}")
        
        return {
            'fold_id': fold_id,
            'best_val_loss': best_val_loss,
            'model_state': best_state,
        }


# Initialize training module
train_config = TrainingConfig()
trainer = TrainingModule(model, preprocessor, train_config, DEVICE)

print(f"Training Configuration:")
for k, v in train_config.to_dict().items():
    print(f"  {k}: {v}")

## Part 5: Cross-Validation Pipeline

In [ ]:
class CrossValidationModule:
    """Manages K-fold cross-validation."""
    
    def __init__(
        self,
        trainer: TrainingModule,
        preprocessor: PreprocessingModule,
        n_splits: int = 3,
        random_state: int = 42,
    ):
        self.trainer = trainer
        self.preprocessor = preprocessor
        self.n_splits = n_splits
        self.random_state = random_state
        self.fold_results = []
    
    def run(
        self,
        X_train: np.ndarray,
        Y_train: np.ndarray,
        X_test: np.ndarray,
    ) -> np.ndarray:
        """Run K-fold CV and return averaged test predictions."""
        print(f"\n{'#'*60}")
        print(f"Starting {self.n_splits}-Fold Cross-Validation")
        print(f"{'#'*60}")
        
        # Setup K-fold
        kf = KFold(
            n_splits=self.n_splits,
            shuffle=True,
            random_state=self.random_state,
        )
        
        test_preds_list = []
        
        for fold_id, (tr_idx, va_idx) in enumerate(kf.split(X_train), start=1):
            # Prepare tensors
            X_tr = self.preprocessor.numpy_to_tensor(X_train[tr_idx])
            Y_tr = self.preprocessor.numpy_to_tensor(Y_train[tr_idx])
            X_va = self.preprocessor.numpy_to_tensor(X_train[va_idx])
            Y_va = self.preprocessor.numpy_to_tensor(Y_train[va_idx])
            
            # Train fold
            fold_result = self.trainer.train_fold(X_tr, Y_tr, X_va, Y_va, fold_id)
            self.fold_results.append(fold_result)
            
            # Predict on test
            test_preds = self._predict(X_test)
            test_preds_list.append(test_preds)
        
        # Ensemble predictions
        ensemble_preds = np.mean(test_preds_list, axis=0)
        
        print(f"\n{'#'*60}")
        print(f"Cross-Validation Complete")
        print(f"Ensemble predictions shape: {ensemble_preds.shape}")
        print(f"Mean fold val loss: {np.mean([f['best_val_loss'] for f in self.fold_results]):.6f}")
        print(f"{'#'*60}")
        
        return ensemble_preds
    
    @torch.no_grad()
    def _predict(self, X_test: np.ndarray) -> np.ndarray:
        """Predict on test set."""
        self.trainer.model.eval()
        X_test_tensor = self.preprocessor.numpy_to_tensor(X_test)
        
        test_loader = DataLoader(
            TensorDataset(X_test_tensor),
            batch_size=self.trainer.config.batch_size,
            shuffle=False,
        )
        
        preds = []
        for (x_vec,) in test_loader:
            A = self.preprocessor.vec_batch_to_adj(x_vec, self.preprocessor.n_lr)
            X_nodes = self.preprocessor.get_node_features(A)
            pred = self.trainer.model(A, X_nodes)
            preds.append(pred.cpu().numpy())
        
        return np.vstack(preds)


# Initialize CV module
cv_module = CrossValidationModule(
    trainer,
    preprocessor,
    n_splits=3,
    random_state=42,
)

print("✓ Cross-validation module initialized")

## Part 6: Run Full Pipeline

In [ ]:
# Execute cross-validation
test_predictions = cv_module.run(X_lr_train, Y_hr_train, X_lr_test)

print(f"\nPredictions Summary:")
print(f"  Shape: {test_predictions.shape}")
print(f"  Min: {test_predictions.min():.6f}")
print(f"  Max: {test_predictions.max():.6f}")
print(f"  Mean: {test_predictions.mean():.6f}")
print(f"  Std: {test_predictions.std():.6f}")

## Part 7: Inference & Submission Module

In [ ]:
class SubmissionModule:
    """Handles submission file generation and validation."""
    
    def __init__(self, repo_root: Path):
        self.repo_root = Path(repo_root)
        self.submission_dir = self.repo_root
    
    def save(self, predictions: np.ndarray, filename: str = 'submission.csv') -> Path:
        """Save predictions to CSV."""
        output_path = self.submission_dir / filename
        np.savetxt(output_path, predictions, delimiter=",")
        return output_path
    
    def validate(self, predictions: np.ndarray, expected_shape: Tuple) -> bool:
        """Validate prediction format."""
        if predictions.shape != expected_shape:
            print(f"❌ Shape mismatch: {predictions.shape} vs {expected_shape}")
            return False
        
        if np.any(np.isnan(predictions)) or np.any(np.isinf(predictions)):
            print(f"❌ Contains NaN or Inf values")
            return False
        
        print(f"✓ Predictions valid")
        return True
    
    def load_and_verify(self, filepath: Path) -> np.ndarray:
        """Load and verify saved submission."""
        preds = np.loadtxt(filepath, delimiter=",")
        if preds.ndim == 1:
            preds = preds[None, :]
        return preds


# Initialize submission module
submission_module = SubmissionModule(repo_root)
print("✓ Submission module initialized")

In [ ]:
# Save submission
output_path = submission_module.save(test_predictions, 'submission.csv')
print(f"\nSubmission saved to: {output_path}")

# Validate
submission_module.validate(test_predictions, (112, 35778))

# Verify saved file
verified_preds = submission_module.load_and_verify(output_path)
print(f"\nFile verification:")
print(f"  Saved shape: {verified_preds.shape}")
print(f"  Match: {np.allclose(test_predictions, verified_preds)}")

## Part 8: Final Report

In [ ]:
class ReportModule:
    """Generates experiment summary."""
    
    def __init__(
        self,
        model_config: ModelConfig,
        train_config: TrainingConfig,
        cv_results: list,
        X_train_shape: Tuple,
        X_test_shape: Tuple,
    ):
        self.model_config = model_config
        self.train_config = train_config
        self.cv_results = cv_results
        self.X_train_shape = X_train_shape
        self.X_test_shape = X_test_shape
    
    def generate(self) -> str:
        """Generate report string."""
        report = []
        report.append("\n" + "="*70)
        report.append("NEURORES-GNN: MODULAR ML PIPELINE - EXPERIMENT REPORT")
        report.append("="*70)
        
        report.append("\n[DATA]")
        report.append(f"  Training samples: {self.X_train_shape[0]}")
        report.append(f"  Test samples: {self.X_test_shape[0]}")
        report.append(f"  LR graph size: 160 × 160")
        report.append(f"  HR graph size: 268 × 268")
        
        report.append("\n[MODEL]")
        report.append(f"  Architecture: GCN Encoder + Cross-Attention Decoder")
        report.append(f"  Hidden dimension: {self.model_config.d_model}")
        report.append(f"  GCN layers: {self.model_config.gcn_layers}")
        report.append(f"  Attention heads: {self.model_config.attn_heads}")
        report.append(f"  Dropout: {self.model_config.dropout}")
        
        report.append("\n[TRAINING]")
        report.append(f"  Epochs: {self.train_config.epochs}")
        report.append(f"  Batch size: {self.train_config.batch_size}")
        report.append(f"  Learning rate: {self.train_config.lr}")
        report.append(f"  Weight decay: {self.train_config.weight_decay}")
        report.append(f"  N-fold CV: {self.train_config.n_folds}")
        
        report.append("\n[CROSS-VALIDATION RESULTS]")
        for i, result in enumerate(self.cv_results, 1):
            report.append(f"  Fold {i} best val loss: {result['best_val_loss']:.6f}")
        
        mean_loss = np.mean([r['best_val_loss'] for r in self.cv_results])
        report.append(f"  Mean CV loss: {mean_loss:.6f}")
        
        report.append("\n[OUTPUT]")
        report.append(f"  Submission: submission.csv")
        report.append(f"  Format: {self.X_test_shape[0]} samples × {35778} features")
        
        report.append("\n" + "="*70)
        report.append("✓ Pipeline completed successfully")
        report.append("="*70 + "\n")
        
        return "\n".join(report)


# Generate report
reporter = ReportModule(
    model_config,
    train_config,
    cv_module.fold_results,
    X_lr_train.shape,
    X_lr_test.shape,
)

print(reporter.generate())